In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MadMaxD") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()

In [ ]:
jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")

shops_df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("user", db_user) \
            .option("password", db_password) \
            .option("dbtable", "public.shops") \
            .option("fetchsize", 1000) \
            .option("driver", "org.postgresql.Driver") \
            .load()

shop_tz_df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("user", db_user) \
            .option("password", db_password) \
            .option("dbtable", "public.shop_timezone") \
            .option("fetchsize", 1000) \
            .option("driver", "org.postgresql.Driver") \
            .load()

# Регистрируем DataFrame-ы как временные вьюхи
shops_df.createOrReplaceTempView("shops")
shop_tz_df.createOrReplaceTempView("shop_timezone")

### Описание результата
1. Сформировать витрину с ID магазина, название магазина и со всем временными зонами присутствия магазина [st_od, shop_name, time_zone_n]
2. Временные зоны time_zone_n сформировать в цифровом формате 0, 1, 2...
3. Если данных нет в time_zone источника, то проставить 3 в time_zone_n
4. Удалить полные дубли строк

In [ ]:
spark.sql("""
          select st_od, shop_name, time_zone_n 
          from (
          select distinct s.st_id as st_od, s.shop_name as shop_name, coalesce(int(replace(st.time_zone, 'RUS', '')), 3) as time_zone_n 
          from shops s 
           join shop_timezone st 
           on s.st_id = st.plant) _
          order by shop_name, st_od, time_zone_n""").show()

In [ ]:
spark.stop()